In [ ]:
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame

code = 'US'

csv_df = pd.read_csv(f'open-meteo/dataset/{code}.csv', sep = ',')

csv_df['city'] = csv_df['city'].str.split(', ').str.get(0)
csv_df.rename(columns={'city': 'name'}, inplace=True)
df2 = csv_df[['date', 'temperature_2m_mean', 'precipitation_sum', 'soil_moisture_7_to_28cm_mean', 'name']]

df3 = df2[ df2['date'] <= '2025-01-01']
df3.to_csv(f'dataset/{code}_2015-2024.csv', index=False)

df4 = df2[ (df2['date'] >= '2025-01-01') & (df2['date'] < '2025-08-01') ]
df4.to_csv(f'dataset/{code}_202501-202507.csv', index=False)


In [1]:
from src import meteo_config_im, meteo_config_aure, meteo_config_cba, meteo_config_cn, meteo_config_us

countries_list = [
    meteo_config_im.countries,
    meteo_config_aure.countries,
    meteo_config_cba.countries,
    meteo_config_cn.countries,
    meteo_config_us.countries
]
a, b, c = 0, 0, 0
for countries in countries_list:
    for country in countries:
        a += 1
        for city in country['city_list']:
            b +=1
            c +=  len(city['latitude'])
print(a, b, c)

11 61 255


In [ ]:
import pandas as pd
import src.charts as charts

a = pd.read_csv('dataset/ID_2011-2020.csv', sep = ',')
b = pd.read_csv('dataset/ID_2021-2024.csv', sep = ',')
c = pd.read_csv('dataset/ID_202501-202508.csv', sep = ',')

df = pd.concat([a, b, c])
# df
df = df[ df['name'] == 'Riau' ]
df
# #
charts.day_annul_plot(df, 'soil_moisture_28_to_100cm_mean', std=1)
charts.day_annul_plot(df, 'soil_moisture_28_to_100cm_mean')


In [ ]:
import pandas as pd

CN_2014_2024 = pd.read_csv('dataset/CN_2014-2024.csv', sep = ',')
CN_2014_2024 =  CN_2014_2024[ CN_2014_2024['name'] == 'Heilongjiang']
CN_2014_2024

In [ ]:
import pandas as pd
def xxx(df, int):
    df['date'] = pd.to_datetime(df['date'])
    df['old_year'] = df['date'].dt.year
    df['old_day_of_year'] = df['date'].dt.dayofyear
    df['new_day_of_year'] = (df['old_day_of_year'] + (365-int)) % 365 + 1
    # 如果 old_day_of_year > new_day_of_year 则 new_year = old_year + 1 否责 new_year = old_year
    df['new_year'] = df['old_year']
    df.loc[df['old_day_of_year'] > df['new_day_of_year'], 'new_year'] = df['old_year'] + 1
    # df['new_year_label'] = (df['new_year']-1).astype(str)  + '~' + df['new_year'].astype(str)
    return df
df = xxx(pd.read_csv('dataset/CN_2014-2024.csv', sep = ','),90)
df

In [ ]:
import matplotlib.font_manager as fm

# 查看所有可用字体
fonts = [f.name for f in fm.fontManager.ttflist]
chinese_fonts = [f for f in fonts if any(char in f for char in ['黑体', '宋体', '微软雅黑', 'Sim'])]

print("可用中文字体:", chinese_fonts)

In [ ]:


import sqlite3
import pickle
from requests import Response  # 用于类型提示

# 连接数据库
conn = sqlite3.connect('.cache.sqlite')
cursor = conn.cursor()
# 查询数据
cursor.execute("SELECT key, value, expires FROM responses LIMIT 1")
row = cursor.fetchone()
key, value_blob, expires = row

# 反序列化value列（跳过开头的0x）
response = pickle.loads(value_blob)  # type: Response

# 获取响应内容
print("URL:", response['url'])
print("状态码:", response['status_code'])
print("头部:", response['headers'])
print("内容:", response['_content']  )# 或 response.json() 如果是JSON




In [ ]:
import zlib
# decompressed_data = zlib.decompress(response['_content'])
a=response['_content'].decode('utf-8')

In [13]:
import pandas as pd
import sqlite3

df  = pd.read_csv('dataset/CN_202501-202508.csv', sep = ',')
conn = sqlite3.connect('example.db')  # 数据库文件名

table_name = 'weather'  # 目标表名

df.to_sql(
    name=table_name,
    con=conn,
    if_exists='replace',  # 处理表已存在的情况：'fail'/'replace'/'append'
    index=False # 不写入DataFrame索引
)

conn.commit()
conn.close()

print(f"数据已成功写入表: {table_name}")

DatabaseError: Execution failed on sql 'DROP TABLE "weather"': database is locked

In [3]:
import pandas as pd

CN_2014_2024 = pd.read_csv('dataset/CN_2014-2024.csv', sep = ',')
CN_2014_2024 = CN_2014_2024[ CN_2014_2024['name'] == 'Heilongjiang']
CN_2014_2024

,date,temperature_2m_mean,precipitation_sum,soil_moisture_7_to_28cm_mean,name
0,2014-01-01 00:00:00+00:00,-18.461153,0.000000,0.346646,Heilongjiang
1,2014-01-02 00:00:00+00:00,-20.187195,0.283333,0.346257,Heilongjiang
2,2014-01-03 00:00:00+00:00,-23.429903,0.000000,0.346069,Heilongjiang
3,2014-01-04 00:00:00+00:00,-23.893442,0.000000,0.345910,Heilongjiang
4,2014-01-05 00:00:00+00:00,-23.218790,0.000000,0.345799,Heilongjiang
...,...,...,...,...,...
32139,2024-12-27 00:00:00+00:00,0.738417,0.000000,0.265896,Anhui
32140,2024-12-28 00:00:00+00:00,0.317583,0.000000,0.261708,Anhui
32141,2024-12-29 00:00:00+00:00,2.885292,0.000000,0.255229,Anhui
32142,2024-12-30 00:00:00+00:00,3.343625,0.000000,0.236708,Anhui


In [11]:
import sqlite3
# 连接数据库
conn = sqlite3.connect('example.db')
cursor = conn.cursor()
# 查询数据
a = cursor.execute("SELECT * FROM weather")
conn.close()


In [2]:
import sqlite3
import pandas as pd

# 连接到 SQLite 数据库（如果数据库不存在，会自动创建）
conn = sqlite3.connect('example.db')  # 替换为你的数据库路径

# 使用 pandas 读取 SQL 查询结果到 DataFrame
query = "SELECT * FROM weather"  # 替换为你的表名
df = pd.read_sql_query(query, conn)

# 关闭连接
conn.close()

# 查看 DataFrame 的前几行